# Notebook complet — Dataset médical, qualité, LoRA et graphiques

Ce notebook couvre toutes les missions :

- tests de qualité des conversations ;
- analyse et nettoyage du dataset médical ;
- préparation des données pour le fine-tuning LoRA ;
- validation de la qualité des conversations médicales ;
- génération des livrables ;
- représentations graphiques avec Matplotlib et Seaborn.


In [ ]:
# ============================================================
# Projet : Qualité des données médicales + préparation LoRA

In [ ]:
# ============================================================
# Missions couvertes :
# - Tests de qualité des conversations
# - Analyse et nettoyage du dataset médical
# - Préparation des données pour le fine-tuning LoRA
# - Validation de la qualité des conversations médicales
#
# Livrables générés :
# - Dataset nettoyé en CSV et Parquet
# - Dataset LoRA en CSV 
# - Split train/test 80/20
# - Rapport qualité en CSV 
# - Graphiques de qualité avec Matplotlib et Seaborn

In [ ]:
# ============================================================

import pandas as pd
import re
from pathlib import Path
from datetime import datetime

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:

DATA_DIR = Path("medical_dataset")
INPUT_FILE = DATA_DIR / "dialogues.parquet"

OUTPUT_DIR = DATA_DIR / "livrables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIGURES_DIR = OUTPUT_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_FILE.exists():
    found_files = list(Path(".").rglob("dialogues.parquet"))

    if len(found_files) == 0:
        raise FileNotFoundError(
            "Impossible de trouver dialogues.parquet. "
            "Vérifie que le fichier existe dans le dossier medical_dataset."
        )
    else:
        INPUT_FILE = found_files[0]
        print("Fichier trouvé automatiquement :", INPUT_FILE)

print("Fichier utilisé :", INPUT_FILE)
print("Dossier des livrables :", OUTPUT_DIR)
print("Dossier des graphiques :", FIGURES_DIR)

In [ ]:

df = pd.read_parquet(INPUT_FILE)

print("\n===== CHARGEMENT DU DATASET =====")
print("Dataset chargé avec succès.")
print("Nombre de lignes :", len(df))
print("Nombre de colonnes :", len(df.columns))
print("Colonnes initiales :", df.columns.tolist())

display(df.head())

In [ ]:
# ============================================================
# 4. Standardisation des noms de colonnes

In [ ]:
# 

print("\n------ Standardisation des noms de colonnes --------")

# Mettre les noms de colonnes en minuscules et supprimer les espaces
df.columns = df.columns.str.strip().str.lower()

print("Colonnes après standardisation :")
print(df.columns.tolist())

required_columns = ["description", "patient", "doctor"]
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Colonnes manquantes dans le dataset : {missing_columns}")
else:
    print("Toutes les colonnes nécessaires sont présentes.")

In [ ]:
# ============================================================
# 5. Fonctions de contrôle qualité

In [ ]:
# ============================================================

# Seuils utilisés pour valider les conversations
MIN_PATIENT_LEN = 20
MIN_DOCTOR_LEN = 30
MAX_PATIENT_LEN = 4000
MAX_DOCTOR_LEN = 6000


def is_empty_series(series):
    """Retourne True si la valeur est NaN ou vide après suppression des espaces."""
    return series.isna() | (series.astype(str).str.strip() == "")


def clean_text(text):
    """
    Nettoyage simple du texte :
    - remplacement des NaN par une chaîne vide
    - passage en minuscules
    - suppression des retours à la ligne et tabulations
    - remplacement des espaces multiples
    - suppression des espaces au début et à la fin
    """
    if pd.isna(text):
        return ""

    text = str(text)
    text = text.lower()
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")
    text = re.sub(r"\s+", " ", text)
    text = text.strip()

    return text


def build_quality_report(data, step_name):
    """
    Construit un rapport qualité sous forme de dictionnaire.
    """
    patient_text = data["patient"].fillna("").astype(str)
    doctor_text = data["doctor"].fillna("").astype(str)

    patient_len = patient_text.str.len()
    doctor_len = doctor_text.str.len()

    report = {
        "etape": step_name,
        "nombre_lignes": len(data),
        "nombre_colonnes": len(data.columns),
        "lignes_completement_vides_nan": int(data.isnull().all(axis=1).sum()),
        "descriptions_vides": int(is_empty_series(data["description"]).sum()),
        "questions_patient_vides": int(is_empty_series(data["patient"]).sum()),
        "reponses_doctor_vides": int(is_empty_series(data["doctor"]).sum()),
        "doublons_exacts": int(data.duplicated().sum()),
        "doublons_patient_doctor": int(data.duplicated(subset=["patient", "doctor"]).sum()),
        "doublons_description_patient_doctor": int(
            data.duplicated(subset=["description", "patient", "doctor"]).sum()
        ),
        "questions_trop_courtes": int((patient_len < MIN_PATIENT_LEN).sum()),
        "reponses_trop_courtes": int((doctor_len < MIN_DOCTOR_LEN).sum()),
        "questions_trop_longues": int((patient_len > MAX_PATIENT_LEN).sum()),
        "reponses_trop_longues": int((doctor_len > MAX_DOCTOR_LEN).sum()),
        "longueur_min_question": int(patient_len.min()) if len(data) > 0 else 0,
        "longueur_moyenne_question": round(float(patient_len.mean()), 2) if len(data) > 0 else 0,
        "longueur_max_question": int(patient_len.max()) if len(data) > 0 else 0,
        "longueur_min_reponse": int(doctor_len.min()) if len(data) > 0 else 0,
        "longueur_moyenne_reponse": round(float(doctor_len.mean()), 2) if len(data) > 0 else 0,
        "longueur_max_reponse": int(doctor_len.max()) if len(data) > 0 else 0,
    }

    return report

In [ ]:
# ============================================================
# 6. Rapport qualité initial

In [ ]:
# ============================================================

rapport_initial = build_quality_report(df, "avant_nettoyage")
rapport_initial_df = pd.DataFrame([rapport_initial])

print("\n===== RAPPORT QUALITÉ INITIAL =====")
display(rapport_initial_df)

In [ ]:
# ============================================================
# 7. Nettoyage du texte

In [ ]:
# ============================================================

print("\n------ Nettoyage du texte --------")

text_columns = ["description", "patient", "doctor"]

for col in text_columns:
    df[col] = df[col].apply(clean_text)

print("Nettoyage du texte terminé.")
display(df.head())

In [ ]:
# ============================================================
# 8. Suppression des conversations vides

In [ ]:
# ============================================================

print("\n------ Suppression des conversations vides --------")

before_empty_filter = len(df)

df = df[
    (df["patient"].str.strip() != "") &
    (df["doctor"].str.strip() != "")
].copy()

after_empty_filter = len(df)
removed_empty_rows = before_empty_filter - after_empty_filter

print("Lignes avant suppression :", before_empty_filter)
print("Lignes après suppression :", after_empty_filter)
print("Lignes supprimées :", removed_empty_rows)

In [ ]:
# ============================================================
# 9. Tests de qualité des conversations

In [ ]:
# ============================================================

print("\n------ Tests de qualité des conversations --------")

df["patient_len"] = df["patient"].apply(len)
df["doctor_len"] = df["doctor"].apply(len)

df["qualite_question_ok"] = df["patient_len"].between(MIN_PATIENT_LEN, MAX_PATIENT_LEN)
df["qualite_reponse_ok"] = df["doctor_len"].between(MIN_DOCTOR_LEN, MAX_DOCTOR_LEN)
df["conversation_valide"] = df["qualite_question_ok"] & df["qualite_reponse_ok"]

quality_test_summary = {
    "conversations_total_avant_filtrage": len(df),
    "questions_valides": int(df["qualite_question_ok"].sum()),
    "reponses_valides": int(df["qualite_reponse_ok"].sum()),
    "conversations_valides": int(df["conversation_valide"].sum()),
    "conversations_rejetees": int((~df["conversation_valide"]).sum()),
}

quality_test_df = pd.DataFrame([quality_test_summary])

display(quality_test_df)

print("Exemples de conversations rejetées :")
display(df[~df["conversation_valide"]].head())

In [ ]:
# ============================================================
# 10. Filtrage des conversations invalides

In [ ]:
# ============================================================

print("\n------ Filtrage des conversations invalides --------")

before_quality_filter = len(df)

rejected_conversations_df = df[~df["conversation_valide"]].copy()
df = df[df["conversation_valide"]].copy()

after_quality_filter = len(df)
removed_quality_rows = before_quality_filter - after_quality_filter

print("Lignes avant filtrage qualité :", before_quality_filter)
print("Lignes après filtrage qualité :", after_quality_filter)
print("Lignes supprimées :", removed_quality_rows)

rejected_conversations_path = OUTPUT_DIR / "conversations_rejetees_qualite.csv"

rejected_conversations_df.to_csv(
    rejected_conversations_path,
    index=False,
    encoding="utf-8-sig"
)

print("Conversations rejetées sauvegardées dans :", rejected_conversations_path)

In [ ]:
# ============================================================
# 11. Suppression des doublons

In [ ]:
# ============================================================

print("\n------ Suppression des doublons --------")

before_duplicates_filter = len(df)

df = df.drop_duplicates(subset=["description", "patient", "doctor"]).copy()

after_duplicates_filter = len(df)
removed_duplicate_rows = before_duplicates_filter - after_duplicates_filter

print("Lignes avant suppression des doublons :", before_duplicates_filter)
print("Lignes après suppression des doublons :", after_duplicates_filter)
print("Doublons supprimés :", removed_duplicate_rows)

In [ ]:
# ============================================================
# 12. Rapport qualité final

In [ ]:
# ============================================================

rapport_final = build_quality_report(df, "apres_nettoyage")
rapport_final_df = pd.DataFrame([rapport_final])

print("\n===== RAPPORT QUALITÉ FINAL =====")
display(rapport_final_df)

print("Aperçu du dataset nettoyé :")
display(df.head())

In [ ]:
# ============================================================
# 13. Validation finale de la qualité

In [ ]:
# ============================================================

print("\n------ Validation finale de la qualité --------")

final_checks = {
    "questions_patient_vides": int((df["patient"].str.strip() == "").sum()),
    "reponses_doctor_vides": int((df["doctor"].str.strip() == "").sum()),
    "doublons_description_patient_doctor": int(
        df.duplicated(subset=["description", "patient", "doctor"]).sum()
    ),
    "questions_trop_courtes": int((df["patient_len"] < MIN_PATIENT_LEN).sum()),
    "reponses_trop_courtes": int((df["doctor_len"] < MIN_DOCTOR_LEN).sum()),
    "questions_trop_longues": int((df["patient_len"] > MAX_PATIENT_LEN).sum()),
    "reponses_trop_longues": int((df["doctor_len"] > MAX_DOCTOR_LEN).sum()),
}

validation_df = pd.DataFrame([
    {
        "controle": key,
        "nombre_problemes": value,
        "statut": "OK" if value == 0 else "A corriger"
    }
    for key, value in final_checks.items()
])

display(validation_df)

if validation_df["nombre_problemes"].sum() == 0:
    print("Validation réussie : le dataset final respecte les règles de qualité.")
else:
    print("Attention : certains contrôles qualité ne sont pas encore validés.")

In [ ]:
# ============================================================
# 14. Préparation des données pour fine-tuning LoRA

In [ ]:
# ============================================================

print("\n------ Préparation LoRA --------")

def build_lora_input(row):
    description = row["description"]
    patient = row["patient"]

    if description.strip() != "":
        return f"medical context: {description}\npatient question: {patient}"
    else:
        return f"patient question: {patient}"


lora_df = pd.DataFrame({
    "instruction": "You are a medical assistant. Answer the patient's question clearly and carefully.",
    "input": df.apply(build_lora_input, axis=1),
    "output": df["doctor"]
})

print("Dataset LoRA créé avec succès.")
print("Shape du dataset LoRA :", lora_df.shape)

display(lora_df.head())

In [ ]:
# ============================================================
# 15. Split train/test 80/20

In [ ]:
# ============================================================

print("\n------ Split train/test 80/20 --------")

# Mélange des index
shuffled_indices = df.sample(frac=1, random_state=42).index

# Position de séparation
split_position = int(0.8 * len(shuffled_indices))

train_indices = shuffled_indices[:split_position]
test_indices = shuffled_indices[split_position:]

# Dataset nettoyé train/test
train_df = df.loc[train_indices].reset_index(drop=True)
test_df = df.loc[test_indices].reset_index(drop=True)

# Dataset LoRA train/test
lora_train_df = lora_df.loc[train_indices].reset_index(drop=True)
lora_test_df = lora_df.loc[test_indices].reset_index(drop=True)

print("Dataset complet nettoyé :", df.shape)
print("Train nettoyé :", train_df.shape)
print("Test nettoyé :", test_df.shape)
print("Train LoRA :", lora_train_df.shape)
print("Test LoRA :", lora_test_df.shape)

In [ ]:
# ============================================================
# 16. Représentations graphiques

In [ ]:
# ============================================================

print("\n------ Création des représentations graphiques --------")

sns.set_theme(style="whitegrid")


# ------------------------------
# Graphique 1 : évolution du nombre de lignes
# ------------------------------

etapes = [
    "Dataset initial",
    "Après suppression vides",
    "Après filtrage qualité",
    "Dataset final"
]

valeurs = [
    rapport_initial["nombre_lignes"],
    after_empty_filter,
    after_quality_filter,
    rapport_final["nombre_lignes"]
]

plt.figure(figsize=(10, 6))
sns.barplot(x=etapes, y=valeurs)

plt.title("Évolution du nombre de lignes après nettoyage")
plt.xlabel("Étapes du traitement")
plt.ylabel("Nombre de lignes")
plt.xticks(rotation=20)

for i, value in enumerate(valeurs):
    plt.text(i, value, str(value), ha="center", va="bottom")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "evolution_nombre_lignes.png", dpi=300)
plt.show()


# ------------------------------
# Graphique 2 : lignes supprimées par type
# ------------------------------

types_suppression = [
    "Conversations vides",
    "Qualité insuffisante",
    "Doublons"
]

valeurs_suppression = [
    removed_empty_rows,
    removed_quality_rows,
    removed_duplicate_rows
]

plt.figure(figsize=(9, 6))
sns.barplot(x=types_suppression, y=valeurs_suppression)

plt.title("Nombre de lignes supprimées par type de problème")
plt.xlabel("Type de problème")
plt.ylabel("Nombre de lignes supprimées")
plt.xticks(rotation=20)

for i, value in enumerate(valeurs_suppression):
    plt.text(i, value, str(value), ha="center", va="bottom")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "lignes_supprimees_par_type.png", dpi=300)
plt.show()


# ------------------------------
# Graphique 3 : distribution des longueurs des questions patient
# ------------------------------

plt.figure(figsize=(10, 6))
sns.histplot(df["patient_len"], bins=50, kde=True)

plt.title("Distribution des longueurs des questions patient")
plt.xlabel("Longueur de la question en caractères")
plt.ylabel("Nombre de conversations")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "distribution_longueur_questions.png", dpi=300)
plt.show()


# ------------------------------
# Graphique 4 : distribution des longueurs des réponses doctor
# ------------------------------

plt.figure(figsize=(10, 6))
sns.histplot(df["doctor_len"], bins=50, kde=True)

plt.title("Distribution des longueurs des réponses doctor")
plt.xlabel("Longueur de la réponse en caractères")
plt.ylabel("Nombre de conversations")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "distribution_longueur_reponses.png", dpi=300)
plt.show()


# ------------------------------
# Graphique 5 : comparaison question patient / réponse doctor
# ------------------------------

longueurs_df = df[["patient_len", "doctor_len"]].copy()

plt.figure(figsize=(9, 6))
sns.boxplot(data=longueurs_df)

plt.title("Comparaison des longueurs des questions et réponses")
plt.xlabel("Type de texte")
plt.ylabel("Longueur en caractères")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "comparaison_longueurs_patient_doctor.png", dpi=300)
plt.show()


# ------------------------------
# Graphique 6 : validation finale qualité
# ------------------------------

plt.figure(figsize=(11, 6))
sns.barplot(
    data=validation_df,
    x="controle",
    y="nombre_problemes"
)

plt.title("Validation finale des contrôles qualité")
plt.xlabel("Contrôles qualité")
plt.ylabel("Nombre de problèmes détectés")
plt.xticks(rotation=45, ha="right")

for i, value in enumerate(validation_df["nombre_problemes"]):
    plt.text(i, value, str(value), ha="center", va="bottom")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "validation_finale_qualite.png", dpi=300)
plt.show()


# ------------------------------
# Graphique 7 : répartition train/test
# ------------------------------

split_df = pd.DataFrame({
    "dataset": ["Train", "Test"],
    "nombre_lignes": [len(train_df), len(test_df)]
})

plt.figure(figsize=(7, 6))
sns.barplot(data=split_df, x="dataset", y="nombre_lignes")

plt.title("Répartition du dataset nettoyé en train/test")
plt.xlabel("Partie du dataset")
plt.ylabel("Nombre de lignes")

for i, value in enumerate(split_df["nombre_lignes"]):
    plt.text(i, value, str(value), ha="center", va="bottom")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "repartition_train_test.png", dpi=300)
plt.show()


# ------------------------------
# Graphique 8 : répartition LoRA train/test
# ------------------------------

lora_split_df = pd.DataFrame({
    "dataset": ["Train LoRA", "Test LoRA"],
    "nombre_lignes": [len(lora_train_df), len(lora_test_df)]
})

plt.figure(figsize=(7, 6))
sns.barplot(data=lora_split_df, x="dataset", y="nombre_lignes")

plt.title("Répartition du dataset LoRA en train/test")
plt.xlabel("Partie du dataset LoRA")
plt.ylabel("Nombre de lignes")

for i, value in enumerate(lora_split_df["nombre_lignes"]):
    plt.text(i, value, str(value), ha="center", va="bottom")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "repartition_lora_train_test.png", dpi=300)
plt.show()


print("Graphiques sauvegardés :")
for fig in FIGURES_DIR.glob("*.png"):
    print("-", fig)

In [ ]:
# ============================================================
# 17. Sauvegarde des livrables

In [ ]:
print("\n------ Sauvegarde des livrables --------")

# Dataset nettoyé complet
cleaned_parquet_path = OUTPUT_DIR / "dialogues_cleaned.parquet"
cleaned_csv_path = OUTPUT_DIR / "dialogues_cleaned.csv"

df.to_parquet(cleaned_parquet_path, index=False)
df.to_csv(cleaned_csv_path, index=False, encoding="utf-8-sig")

# Dataset LoRA complet en CSV seulement
lora_csv_path = OUTPUT_DIR / "dialogues_lora.csv"

lora_df.to_csv(
    lora_csv_path,
    index=False,
    encoding="utf-8-sig"
)

# Train/Test cleaned
train_cleaned_path = OUTPUT_DIR / "train_cleaned.csv"
test_cleaned_path = OUTPUT_DIR / "test_cleaned.csv"

train_df.to_csv(
    train_cleaned_path,
    index=False,
    encoding="utf-8-sig"
)

test_df.to_csv(
    test_cleaned_path,
    index=False,
    encoding="utf-8-sig"
)

# Train/Test LoRA
train_lora_path = OUTPUT_DIR / "train_lora.csv"
test_lora_path = OUTPUT_DIR / "test_lora.csv"

lora_train_df.to_csv(
    train_lora_path,
    index=False,
    encoding="utf-8-sig"
)

lora_test_df.to_csv(
    test_lora_path,
    index=False,
    encoding="utf-8-sig"
)

print("Fichiers sauvegardés :")
print("-", cleaned_parquet_path)
print("-", cleaned_csv_path)
print("-", lora_csv_path)
print("-", train_cleaned_path)
print("-", test_cleaned_path)
print("-", train_lora_path)
print("-", test_lora_path)